# Notebook 5: Sequence Logos and Conservation Analysis

**Pipeline step:** Convert the active site CSV tables from Step 4 into sequence logos — a visual summary of amino acid conservation at each active site position.

**Overview:** A sequence logo encodes, at each position, both which amino acids appear and how conserved they are. In this final notebook you will understand the information-theory basis of logos, build them from the active site CSVs using Logomaker, and interpret the biological meaning of each conserved position.

---

## Learning objectives

- Understand what a sequence logo is and how it differs from a simple alignment
- Know the difference between frequency and information content
- Understand what bits mean in this context
- Convert an active site CSV into a frequency matrix using pandas
- Use Logomaker to generate information-content logos
- Walk through `make_logos.py` line by line
- Interpret the biological meaning of conserved positions in the AMP site, N-OH site, and cupin ZN site
- Customise logo appearance: colour schemes, labels, figure size

---

## 1. What is a sequence logo?

A sequence logo is a graphical representation of a sequence alignment. At each position:
- The **height of the entire column** represents how conserved that position is (more conserved = taller)
- The **height of each letter** within the column represents that amino acid's frequency
- The **letters are stacked** with the most frequent at the top

Compared to printing the raw alignment, a logo immediately shows:
- Which positions are under strong selection (tall = functionally important)
- What substitutions are tolerated at variable positions

> **Key concept:** A tall, single-letter column (e.g. a column that is always histidine) means that residue is likely essential for function. Mutating it would probably break the enzyme. A short column with many different letters means that position tolerates substitutions — it is not critical for function.

---

## 2. Frequency vs information content

### Frequency matrix

A frequency matrix records, for each position, the fraction of sequences that contain each amino acid:

| Position | A | C | ... | H | ... | S | T | ...
|---|---|---|---|---|---|---|---|---|
| S138 | 0 | 0 | ... | 0 | ... | 0.59 | 0.35 | ...|
| T141 | 0 | 0 | ... | 0 | ... | 0 | 1.00 | ...|

T141 is threonine in every enzyme → frequency of T = 1.0.

### Information content

For amino acids, a completely random position would have 20 equally likely residues: probability 1/20 = 0.05 each. That corresponds to **~4.32 bits** of maximum possible information.

The information content at position $i$ is:

$$R_i = \log_2(20) + \sum_{a} f_{i,a} \log_2(f_{i,a})$$

where $f_{i,a}$ is the frequency of amino acid $a$ at position $i$.

**Intuition:**
- $R_i = 0$ bits → all 20 amino acids equally likely → no information; nothing is conserved
- $R_i \approx 4.32$ bits → one amino acid at 100% frequency → maximum information; that residue is invariant

In Logomaker, the height of each letter is $f_{i,a} \times R_i$ (information content scaled by frequency).

> **Key concept:** Information content converts frequency into a biologically meaningful scale. A position with T at 100% contributes ~4.32 bits — it tells you exactly which amino acid is required. A position with ten different amino acids each at 10% contributes near 0 bits — random, no constraint.

In [ ]:
# ── Demonstrate information content calculation ────────────────────────
import numpy as np

def information_content(frequencies):
    """
    Calculate R_i (information content in bits) for one position.
    frequencies: list of 20 amino acid frequencies (must sum to 1)
    """
    import math
    max_info = math.log2(20)   # ~4.32 bits for amino acids
    entropy  = 0.0
    for f in frequencies:
        if f > 0:   # log2(0) is undefined; skip zero frequencies
            entropy += f * math.log2(f)
    return max_info + entropy   # R_i = log2(20) - H_i


# Case 1: perfectly conserved position (only T)
# 19 AAs at 0.0, one AA at 1.0
conserved = [0.0] * 19 + [1.0]
print("Fully conserved position: {:.3f} bits  (max ~4.32)".format(
    information_content(conserved)))

# Case 2: completely random (all 20 equally likely)
random_pos = [1/20] * 20
print("Completely random position: {:.3f} bits  (expected 0)".format(
    information_content(random_pos)))

# Case 3: two amino acids equally frequent (50/50)
two_aas = [0.5, 0.5] + [0.0] * 18
print("Two AAs at 50% each: {:.3f} bits".format(
    information_content(two_aas)))

---

## 3. Building a frequency matrix from the active site CSV

In [ ]:
# ── Load the MetRS active site CSV ────────────────────────────────────
import pandas as pd

METRS_CSV = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/results/metrs_active_site.csv"
metrs_df  = pd.read_csv(METRS_CSV)

# 20 canonical amino acids (alphabetical)
AAS = sorted("ACDEFGHIKLMNPQRSTVWY")

# The 11 AMP-site position columns
AMP_COLS = ["S138", "F139", "P140", "T141", "V179", "Q182",
            "A384", "L387", "R390", "N421", "R425"]


def build_frequency_matrix(df, columns):
    """
    Build a frequency matrix from a DataFrame of single-letter residue codes.

    Returns a DataFrame with:
      - rows indexed 0 to len(columns)-1  (one per position)
      - columns = 20 amino acid letters
      - values = frequencies (0.0 to 1.0, summing to 1 per row)
    """
    n = len(df)                     # total number of sequences

    # Count occurrences of each amino acid at each position
    counts = {
        col: {aa: 0 for aa in AAS}  # initialise all counts to 0
        for col in columns
    }

    for _, row in df.iterrows():    # iterate over each enzyme (each row)
        for col in columns:
            aa = str(row[col]).strip()   # single-letter code
            if aa in AAS:               # ignore gaps ('-') and missing ('X')
                counts[col][aa] += 1

    # Convert counts to frequencies
    freq_data = {
        col: {aa: counts[col][aa] / n for aa in AAS}
        for col in columns
    }

    # Build DataFrame: rows = positions, columns = amino acids
    freq_df = pd.DataFrame(freq_data).T    # .T transposes rows and columns
    freq_df.index = range(len(columns))    # integer index for Logomaker

    return freq_df


freq_df = build_frequency_matrix(metrs_df, AMP_COLS)
print("Frequency matrix shape: {} positions x {} amino acids".format(
    freq_df.shape[0], freq_df.shape[1]))
print()

# Show only columns with non-zero frequency to keep output readable
nonzero = freq_df.loc[:, (freq_df > 0).any(axis=0)]
nonzero.index = AMP_COLS   # label rows with position names
print(nonzero.round(2).to_string())

---

## 4. Logomaker

**Logomaker** is a Python library that generates sequence logos from frequency or information matrices. Key functions:

- `logomaker.transform_matrix(freq_df, from_type='probability', to_type='information')` — converts a frequency matrix to an information-content matrix
- `logomaker.Logo(info_df, ax=ax, color_scheme='chemistry')` — draws the logo on a matplotlib axis

Colour schemes:
- `'chemistry'` — groups residues by chemical property (green=polar, blue=basic, red=acidic, black=hydrophobic)
- `'NajafabadiEtAl2017'` — a colour-blind friendly scheme
- Custom colours: pass a dict `{letter: hex_colour}`

In [ ]:
# ── Convert frequency matrix to information-content matrix ────────────
import logomaker

# logomaker requires rows indexed as integers
freq_df_int = build_frequency_matrix(metrs_df, AMP_COLS)

# Transform: probability → information (bits)
info_df = logomaker.transform_matrix(
    freq_df_int,
    from_type="probability",
    to_type="information",
)

print("Information matrix (bits):")
# Show only the dominant amino acid per position for readability
for i, col_name in enumerate(AMP_COLS):
    row       = info_df.loc[i]
    dominant  = row.idxmax()          # amino acid with highest info content
    total_ic  = row.sum()             # total info content at this position
    print("  Position {:>5} (row {:2d})  total IC = {:.3f} bits  dominant: {}".format(
        col_name, i, total_ic, dominant))

In [ ]:
# ── Draw the AMP-site logo ─────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))

# Draw logo: each column = one active site position
logomaker.Logo(
    info_df,
    ax=ax,
    color_scheme="chemistry",   # colour by chemical class
    stack_order="small_on_top", # put minority amino acids above majority
)

ax.set_ylabel("Information (bits)", fontsize=12)
ax.set_title("MetRS AMP-binding site — Hao 2026", fontsize=13, fontweight="bold", pad=10)

# Label x-axis with position names (S138, F139, ...) instead of integers
ax.set_xticks(range(len(AMP_COLS)))
ax.set_xticklabels(AMP_COLS, rotation=45, ha="right", fontsize=10)

ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("/tmp/logo_amp_site_notebook.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/logo_amp_site_notebook.png")

---

## 5. Walking through `make_logos.py`

In [ ]:
# ── Print make_logos.py for reference ─────────────────────────────────
LOGO_SCRIPT = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step5/make_logos.py"

with open(LOGO_SCRIPT) as fh:
    print(fh.read())

**Key points in `make_logos.py`:**

1. **`make_logo(df, columns, title, out_path)`** — a single reusable function that handles all three logos. This is good practice: avoid duplicating code.

2. **The cupin site columns** are read dynamically: `cupin_cols = [c for c in cupin_df.columns if c not in ('Enzyme', 'RMSD', 'Best_model', 'Confidence')]`. This means the script works correctly regardless of how many ZN-proximal residues are detected.

3. **`stack_order='small_on_top'`** puts less frequent amino acids above more frequent ones, which makes the dominant residue the most visible (it is the largest letter at the bottom).

Run the script:

```bash
conda activate comp_analysis
cd /scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step5/
python make_logos.py
```

Output:
- `logo_amp_site.png`
- `logo_noh_site.png`
- `logo_cupin_site.png`

---

## 6. Biological interpretation of the logos

### AMP-binding site

- **T141** — fully conserved (1.0 frequency). The threonine hydroxyl makes a hydrogen bond to the AMP phosphate. Any substitution disrupts this and likely abolishes adenylation activity.
- **S138** — highly conserved (serine or threonine). The –OH side chain contacts the ribose of AMP.
- **R390** — arginine or histidine (both positively charged). The positive charge neutralises the phosphate of AMP — so the positive charge is conserved even though the exact residue varies.
- **A384** — variable. This residue is peripheral to the AMP pocket; substitutions are tolerated.

### N-OH substrate site

- **E284 and E286** — glutamate residues that are highly conserved. These acidic residues likely deprotonate or position the N-hydroxylamine substrate. Conservation here points to a shared catalytic mechanism.
- **D420** — aspartate, conserved alongside N421. The D420-N421 pair forms part of the substrate recognition pocket unique to hydrazine synthetases (absent in canonical MetRS enzymes).

### Cupin ZN site

- **H11, H13, H51** — three histidines. The cupin superfamily characteristically coordinates zinc through a HxHxE (or similar) triad. Histidine is the dominant ZN ligand in cupin proteins, and all three positions show near-perfect conservation.
- **E17** — glutamate, the third ZN ligand in the HxHxE motif. Fully conserved.
- **W19** — tryptophan, partially conserved. This is a structural residue that packs against the ZN-binding loop rather than directly coordinating zinc; some substitutions (to F or Y) are tolerated.

In [ ]:
# ── Generate the N-OH site logo ────────────────────────────────────────
NOH_COLS = ["N143", "E284", "E286", "D420", "N421", "L424"]

freq_noh = build_frequency_matrix(metrs_df, NOH_COLS)
info_noh = logomaker.transform_matrix(
    freq_noh, from_type="probability", to_type="information"
)

fig, ax = plt.subplots(figsize=(8, 4))
logomaker.Logo(info_noh, ax=ax, color_scheme="chemistry", stack_order="small_on_top")
ax.set_ylabel("Information (bits)", fontsize=12)
ax.set_title("MetRS N-OH substrate site — Hao 2026", fontsize=13, fontweight="bold", pad=10)
ax.set_xticks(range(len(NOH_COLS)))
ax.set_xticklabels(NOH_COLS, rotation=45, ha="right", fontsize=10)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("/tmp/logo_noh_site_notebook.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/logo_noh_site_notebook.png")

In [ ]:
# ── Generate the cupin ZN-site logo ───────────────────────────────────
CUPIN_CSV = "/scratch/p318738/SSA/AAR-COMP-012/AAR-COMP-012-Step4/results/cupin_active_site.csv"
cupin_df  = pd.read_csv(CUPIN_CSV)

# The cupin site columns are everything after the 4 metadata columns
cupin_cols = [c for c in cupin_df.columns
              if c not in ("Enzyme", "RMSD", "Best_model", "Confidence")]

print("Cupin ZN-site positions: {}".format(cupin_cols))

freq_cupin = build_frequency_matrix(cupin_df, cupin_cols)
info_cupin = logomaker.transform_matrix(
    freq_cupin, from_type="probability", to_type="information"
)

fig, ax = plt.subplots(figsize=(7, 4))
logomaker.Logo(info_cupin, ax=ax, color_scheme="chemistry", stack_order="small_on_top")
ax.set_ylabel("Information (bits)", fontsize=12)
ax.set_title("Cupin ZN-coordination site (5 A cutoff, PyrN reference)",
             fontsize=13, fontweight="bold", pad=10)
ax.set_xticks(range(len(cupin_cols)))
ax.set_xticklabels(cupin_cols, rotation=45, ha="right", fontsize=10)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("/tmp/logo_cupin_site_notebook.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/logo_cupin_site_notebook.png")

---

## 7. Customising logos

You can control many visual properties of Logomaker logos.

In [ ]:
# ── Custom colour scheme example ──────────────────────────────────────
# Define a colour dictionary: amino acid letter → hex colour
# Here we highlight zinc-binding residues (H, E, C) in blue
custom_colors = {}
zinc_ligands = ["H", "E", "C"]

for aa in AAS:
    if aa in zinc_ligands:
        custom_colors[aa] = "#1f77b4"   # blue for ZN-binding residues
    else:
        custom_colors[aa] = "#aaaaaa"   # grey for all others

fig, ax = plt.subplots(figsize=(7, 4))
logomaker.Logo(
    info_cupin,
    ax=ax,
    color_scheme=custom_colors,   # use our custom colour dict
    stack_order="small_on_top",
    font_name="DejaVu Sans",      # any matplotlib font
)
ax.set_ylabel("Information (bits)", fontsize=12)
ax.set_title("Cupin ZN site — blue = potential ZN ligands",
             fontsize=12, fontweight="bold", pad=10)
ax.set_xticks(range(len(cupin_cols)))
ax.set_xticklabels(cupin_cols, rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.savefig("/tmp/logo_cupin_custom_colours.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/logo_cupin_custom_colours.png")

In [ ]:
# ── Frequency logo (not information) ──────────────────────────────────
# Sometimes you want the raw frequency rather than information content.
# Frequency logos show which amino acids are present without weighting
# by conservation — useful when you want to see minority residues clearly.

# The frequency matrix already exists: freq_cupin
# Just pass it directly to Logomaker without transforming

fig, ax = plt.subplots(figsize=(7, 4))
logomaker.Logo(
    freq_cupin,
    ax=ax,
    color_scheme="chemistry",
    stack_order="small_on_top",
)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Cupin ZN site — frequency logo",
             fontsize=12, fontweight="bold", pad=10)
ax.set_xticks(range(len(cupin_cols)))
ax.set_xticklabels(cupin_cols, rotation=45, ha="right", fontsize=10)
plt.tight_layout()
plt.savefig("/tmp/logo_cupin_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /tmp/logo_cupin_frequency.png")

---

## Summary

In this notebook you:
- Learned what sequence logos display and how to read them
- Understood the distinction between frequency and information content (bits)
- Built a frequency matrix from the Step 4 active site CSVs using pandas
- Used Logomaker to generate information-content logos for three active sites
- Walked through `make_logos.py` and understood every step
- Interpreted the biological significance of conserved positions:
  - T141 and S138 — invariant AMP-recognition residues
  - E284/E286 — conserved catalytic residues for N-OH substrate
  - HxHxE triad (H11, H13, E17) — classic cupin ZN coordination signature
- Customised logo appearance with custom colour schemes and frequency mode

**Congratulations — you have completed the full HS active site analysis pipeline!**

You have:
1. Fetched and curated 24 HS sequences in a structured CSV
2. Separated cupin and MetRS domains into FASTA files
3. Predicted 3D structures with Boltz2 on the Hábrók HPC cluster
4. Aligned all structures to PyrN and extracted active site residues
5. Visualised conservation as sequence logos

---

## Try it yourself

**Exercise 1:** Load `cupin_active_site.csv` and print the frequency of each amino acid at the H11 position. Which amino acid is most common? Is it always histidine?

```python
# Hint: cupin_df['H11'].value_counts()
# your code here
```

**Exercise 2:** Modify `make_logos.py` to also save a **frequency** (not information) logo for the AMP site. Save it as `logo_amp_site_frequency.png` in the same directory.

```python
# Hint: skip the transform_matrix() step and pass freq_df directly to logomaker.Logo()
# Add a second call to make_logo() or write a new function make_freq_logo()
# your code here
```

**Exercise 3:** Make an AMP-site logo using only the 5 enzymes with the lowest RMSD (most structurally similar to PyrN). Do the logos differ from the full-dataset logo? What does this tell you?

```python
# Hint: metrs_df_sorted = metrs_df.sort_values('RMSD')
#        top5 = metrs_df_sorted.head(5)  (or 6 if including PyrN_crystal)
# Then pass top5 to build_frequency_matrix() and re-draw the logo
# your code here
```